In [13]:
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader

In [3]:
#datasets
import torchvision
from torchvision import datasets,transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

trainset=datasets.MNIST(root="./data",download=True,train=True,transform=transform)
testset=datasets.MNIST(root="./data",download=True,train=False,transform=transform)

In [6]:
trainloader=DataLoader(trainset,shuffle=True,batch_size=64)
testloader=DataLoader(testset,batch_size=64)

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(1,28,padding=1,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #14x14

            nn.Conv2d(28,56,padding=1,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #7x7

        )

        self.fc_layers=nn.Sequential(
            nn.Flatten(),
            nn.Linear(56*7*7,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x

In [11]:
model=CNN()

In [12]:
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [13]:
epochs=10
for epoch in range(epochs):
    training_loss=0.0

    for images,labels in trainloader:
        optimizer.zero_grad()

        output=model.forward(images)
        loss=criteria(output,labels)
        loss.backward()
        optimizer.step()

        training_loss+=loss.item()

    print(f"for {epoch+1}/{epochs} & loss={training_loss/len(trainloader)}")

for 1/10 & loss=0.15486953198772743
for 2/10 & loss=0.04682282410099852
for 3/10 & loss=0.03211997146023187
for 4/10 & loss=0.02296171386406679
for 5/10 & loss=0.018309366338931498
for 6/10 & loss=0.014941416884772355
for 7/10 & loss=0.011769322796807131
for 8/10 & loss=0.009623873113774445
for 9/10 & loss=0.007583221266896992
for 10/10 & loss=0.006700413874373148


In [14]:
import torch
model.eval()
correct=0.0
total=0.0
with torch.no_grad():
    for images,labels in testloader:
        output=model.forward(images)
        _,predicted=torch.max(output,1)
    
        correct+=(predicted==labels).sum().item()
        total+=labels.size(0)

    print(f"accuracy={correct/total *100}")


accuracy=98.89


In [15]:
import torch

In [10]:
#RNN
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        self.fc=nn.Linear(hidden_size,10) #nn.Linear(in_features, out_features)
 
    def forward(self,x):
        # optional => shape (num of layers, batch size, hidden size)
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size,)
        output,_=self.rnn(x,h0) #The output is (batch,sequence,hidden)
        output=self.fc(output[:,-1,:]) #Take the Last Time Step
        return output

In [22]:
input_size=28
model=RNN(input_size)
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [24]:
epochs=10
for epoch in range(epochs):
    model.train()
    for xb,yb in trainloader:
        optimizer.zero_grad()
        xb=xb.squeeze(1)
        output=model(xb)
        #output=torch.sigmoid(output.squeeze())
        loss=criteria(output,yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")


epoch = 1/10 and loss = 0.28092920780181885
epoch = 2/10 and loss = 0.16224126517772675
epoch = 3/10 and loss = 0.2093183547258377
epoch = 4/10 and loss = 0.12363622337579727
epoch = 5/10 and loss = 0.013852658681571484
epoch = 6/10 and loss = 0.02386881411075592
epoch = 7/10 and loss = 0.06628253310918808
epoch = 8/10 and loss = 0.05258026346564293
epoch = 9/10 and loss = 0.17061270773410797
epoch = 10/10 and loss = 0.14561527967453003


In [25]:
model.eval()
correct=0
total=0
with torch.no_grad():
    for xb,yb in testloader:
        xb=xb.squeeze(1)
        output=model(xb)
        _,predicted=torch.max(output.data,1)
        total+=yb.size(0)
        correct+=(predicted==yb).sum().item()
    print(f"accuracy={correct/total *100}")

accuracy=95.6


input_size  
Number of features at each time step.
For MNIST: each row has 28 pixels → input_size=28.

seq_len (sequence length)  
How many time steps the RNN processes.
For MNIST: 28 rows → seq_len=28.

hidden_size  
The size of the hidden state vector at each time step.
Example: hidden_size=128 means each RNN cell outputs a 128‑dimensional vector.
Think of it as the “capacity” or “memory width” of the RNN, not the number of layers.

num_layers  
The number of stacked RNN layers.
Example: num_layers=2 means the output of the first RNN layer is fed into a second RNN layer.
This is the parameter that controls how many hidden layers you have.